In [ ]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from scipy import sparse

from spatialrsp import utils, core

In [ ]:
os.makedirs('figures', exist_ok=True)

In [ ]:
if not os.path.exists("data/kpmp_sn.h5ad"):
    os.makedirs("data", exist_ok=True)
    utils.download_kpmp(data_type="sn", verbose=True)

In [ ]:
kpmp_adata = utils.load_data("data/kpmp_sn.h5ad", verbose=True)
print(kpmp_adata)

In [ ]:
# qc_kpmp_adata = utils.preprocessing.quality_control(
#     adata=kpmp_adata,
#     min_nonzero=0.1,
#     max_dropout=0.5,
#     verbose=True,
# )

qc_kpmp_adata = kpmp_adata
print(qc_kpmp_adata)

In [ ]:
genes_list = qc_kpmp_adata.var["feature_name"].astype(str).tolist()
qc_kpmp_adata.var_names = genes_list

qc_kpmp_adata.var_names_make_unique()
print(qc_kpmp_adata.var_names)

In [ ]:
all_kpmp = qc_kpmp_adata.copy()
normal_kpmp = all_kpmp[all_kpmp.obs["disease"] == "normal"].copy()
ckd_kpmp = all_kpmp[all_kpmp.obs["disease"] == "chronic kidney disease"].copy()
aki_kpmp = all_kpmp[all_kpmp.obs["disease"] == "acute kidney failure"].copy()

In [ ]:
tal_kpmp = all_kpmp[all_kpmp.obs["subclass.l1"] == "TAL"].copy()
tal_normal_kpmp = normal_kpmp[normal_kpmp.obs["subclass.l1"] == "TAL"].copy()
tal_ckd_kpmp = ckd_kpmp[ckd_kpmp.obs["subclass.l1"] == "TAL"].copy()
tal_aki_kpmp = aki_kpmp[aki_kpmp.obs["subclass.l1"] == "TAL"].copy()

In [ ]:
datasets = {
    "Normal": tal_normal_kpmp,
    "CKD": tal_ckd_kpmp,
    "AKI": tal_aki_kpmp,
    "All": tal_kpmp,
}

In [ ]:
# TODO: Figure out how to implement A1 and A2

This is a scrappy implementation of a quick exploratory pipeline that randomly picks a pair of TAL‐expressed genes meeting basic coverage criteria, then visualizes their spatial and angular expression patterns across Normal, CKD, and AKI conditions. First, it highlights the top expressers of each gene on TAL UMAP embeddings using a gray background and red overlays, producing side-by-side panels for each disease state. Next, it computes RSP curves in absolute mode, comparing the foreground (top expressers) to the full background distribution—and renders them as 2x3 polar plots (one row per gene, one column per condition), complete with expected‐foreground and background curves. All figures are saved under a gene-pair–specific folder, making it easy to run many iterations in a batch.

In [ ]:
out_base = "figures/random_gene_pair"
os.makedirs(out_base, exist_ok=True)

In [ ]:
vp = np.asarray(utils.select_vantage_point(tal_kpmp, "X_umap", verbose=False)).ravel()[
    :2
]
coords_2d, polars = utils.load_coords_and_angles(datasets, vp)

In [ ]:
def plot_umap_for_gene(gene, out_dir, top_pct: float = 0.05):
    """
    For `gene`, plot 1x3 UMAP panels (TAL-only cells),
    highlighting the top `top_pct` cells in red on a gray background.

    Saves to out_dir/umap_<gene>.png.
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)
    fig.suptitle(
        f"TAL UMAP — {gene} (top {int(top_pct*100)}% highlighted)", fontsize=16
    )
    plt.subplots_adjust(wspace=0.2, top=0.88)

    for ax, (state, ad) in zip(axes, datasets.items()):
        # 1) extract expression
        mat = ad[:, gene].X
        expr = (
            mat.toarray().ravel() if sparse.issparse(mat) else np.asarray(mat).ravel()
        )

        # 2) determine UMAP coords and threshold
        umap = ad.obsm["X_umap"]
        thr = np.percentile(expr, 100 * (1 - top_pct))
        fg = umap[expr > thr]

        # 3) overlay: background = all points, foreground = high expressers
        core.plot_embedding_overlay(
            bg_coords=umap,
            fg_coords_dict={f"{gene} ≥{thr:.2f}": fg},
            ax=ax,
            title=f"{state} ({ad.n_obs} cells)",
        )

    fn = os.path.join(out_dir, f"umap_{gene}.png")
    plt.savefig(fn, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  • Saved UMAP overlay for {gene} → {fn}")

In [ ]:
def plot_rsp_curves(
    g1,
    g2,
    out_dir,
    pct=0.1,
    scanning_window=np.pi,
    resolution=100,
):
    """
    Compute and plot RSP angular-area curves for gene g1 and g2 across the three states,
    but only fg1 vs bg. Produces a 2x3 grid: rows = [g1, g2], cols = [Normal, CKD, AKI].
    """
    angle_range = np.linspace(0, 2 * np.pi, resolution, endpoint=False)

    expr = {
        st: (utils.extract_expr(ad, g1), utils.extract_expr(ad, g2))
        for st, ad in datasets.items()
    }

    e1_n, e2_n = expr["Normal"]
    fg1_mask_n = utils.select_top(e1_n, pct)
    fg2_mask_n = utils.select_top(e2_n, pct)

    curves_bg = {}
    curves_fg = {g1: {}, g2: {}}
    curves_exp = {g1: {}, g2: {}}

    for state, ad in datasets.items():
        theta = polars[state]

        if state == "Normal":
            m1, m2 = fg1_mask_n, fg2_mask_n
        else:
            m1 = utils.select_top(expr[state][0], pct)
            m2 = utils.select_top(expr[state][1], pct)

        a1, exp1, bg = core.compute_angular_area(
            theta_fg1=theta[m1],
            theta_bg=theta,
            scanning_window=scanning_window,
            resolution=resolution,
            scanning_range=angle_range,
            mode="absolute",
        )

        a2, exp2, _ = core.compute_angular_area(
            theta_fg1=theta[m2],
            theta_bg=theta,
            scanning_window=scanning_window,
            resolution=resolution,
            scanning_range=angle_range,
            mode="absolute",
        )

        curves_bg[state] = bg
        curves_fg[g1][state] = a1
        curves_exp[g1][state] = exp1
        curves_fg[g2][state] = a2
        curves_exp[g2][state] = exp2

    fig, axes = plt.subplots(
        2, 3, subplot_kw={"projection": "polar"}, figsize=(18, 10), sharey=True
    )
    fig.suptitle(
        f"RSP angular-area (top {int(pct*100)}%) — {g1} & {g2}", fontsize=18, y=1.02
    )
    plt.subplots_adjust(wspace=0.3, hspace=0.4)

    for row, gene in enumerate((g1, g2)):
        for col, state in enumerate(("Normal", "CKD", "AKI")):
            ax = axes[row, col]
            core.plot_single_rsp_curve(
                angle_range=angle_range,
                fg_curve=curves_fg[gene][state],
                bg_curve=curves_bg[state],
                expected_fg_curve=curves_exp[gene][state],
                ax=ax,
                label=gene,
            )
            ax.set_title(f"{state} — {gene}")

    outpath = os.path.join(out_dir, f"rsp_polar_fg1_vs_bg_{g1}_{g2}.png")
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  • Saved polar RSP curves (fg vs bg) → {outpath}")

In [ ]:
X = tal_kpmp.X
n_cells, n_genes = X.shape
genes = np.array(tal_kpmp.var_names)

if sparse.isspmatrix_csr(X) or sparse.isspmatrix_csc(X):
    nnz_per_gene = X.getnnz(axis=0)  # very fast in sparse
else:
    nnz_per_gene = np.count_nonzero(X, axis=0)  # fast in dense

coverage = nnz_per_gene / n_cells

mask_valid = (nnz_per_gene >= 30) & (coverage >= 0.10)
valid_genes_all = genes[mask_valid]
print(f"Found {len(valid_genes_all)} genes with ≥30 cells & ≥10% coverage")

if len(valid_genes_all) < 2:
    raise RuntimeError("Not enough valid genes to choose from")

random_subset = np.random.choice(genes, size=1000, replace=False)
valid_in_subset = np.intersect1d(random_subset, valid_genes_all)
print(f"{len(valid_in_subset)} of your 1000 random genes also pass the filter")

if len(valid_in_subset) < 2:
    raise RuntimeError("Too few valid genes in your random draw—try again")

In [ ]:
for i in range(100):
    gene1, gene2 = random.sample(valid_in_subset.tolist(), 2)

    if gene1 == gene2:
        continue  # skip if both genes are the same
    elif os.path.exists(os.path.join(out_base, f"{gene1}_{gene2}")):
        print(f"Skipping already processed pair: {gene1}, {gene2}")
        continue
    elif os.path.exists(os.path.join(out_base, f"{gene2}_{gene1}")):
        print(f"Skipping already processed pair: {gene2}, {gene1}")
        continue

    print(f"▶ Selected gene pair: {gene1}, {gene2}")

    gene_dir = os.path.join(out_base, f"{gene1}_{gene2}")
    os.makedirs(gene_dir, exist_ok=True)

    plot_umap_for_gene(gene1, gene_dir)
    plot_umap_for_gene(gene2, gene_dir)
    plot_rsp_curves(gene1, gene2, gene_dir)
    print(f"  • Plotted UMAP and RSP curves for {gene1} & {gene2} in {gene_dir}")
    print("-" * 40)
    plt.close("all")
    print(f"Finished processing gene pair {i + 1}/10: {gene1}, {gene2}")
    print("-" * 40)
    print("Waiting for a moment before the next pair...")
    plt.pause(1.0 + random.uniform(0, 1))

print("All done.")

For each pair of genes, the script compares their spatial expression patterns across kidney cell subsegments and conditions. It selects the top-expressing cells, projects them into angular space around a common vantage point, computes an RMSD between Normal and disease states, derives p-values via permutation, controls the false discovery rate, and generates summary plots for each segment and percentile threshold.

In [ ]:
def p_to_star(p: float) -> str:
    """
    Convert a p-value into a common 'star' notation for significance.

    Parameters
    ----------
    p : float
        A single p-value.

    Returns
    -------
    str
        '***' if p<0.001, '**' if p<0.01, '*' if p<0.05, 'ns' otherwise,
        or 'NA' if the p-value is NaN.
    """
    if np.isnan(p):
        return "NA"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def compute_segment_stats(
    seg,
    pct,
    datasets,
    coords_2d,
    polars,
    expr,
    gene1,
    gene2,
    vp,
    out_dir,
    min_cells,
    scanning_window,
    resolution,
    angle_range,
    n_reps,
):
    """
    For a given segment and percentile threshold, compute observed and null RMSD
    metrics comparing angular RSPs between Normal and disease states.

    Steps:
      1. Identify the top-pct cells for each gene in the Normal dataset.
      2. Compute their angular response areas a1_n, a2_n and background curve.
      3. Plot the composite Normal RSP figure.
      4. For each disease state (CKD, AKI):
         a. Repeat area computation on top-pct cells.
         b. Plot composite RSP figure for that state.
         c. Compute observed RMSD from Normal reference.
         d. Generate null distribution by permuting cell labels n_reps times.
         e. Derive two-sided p-value.
    5. Return observed values, p-values, and null distribution.

    Parameters
    ----------
    seg : str
        Subsegment label (e.g. 'C-TAL').
    pct : float
        Fraction of top cells to analyze.
    datasets : dict[str, AnnData]
    coords_2d : dict[str, np.ndarray]
    polars : dict[str, np.ndarray]
    expr : dict[str, tuple[np.ndarray, np.ndarray]]
        Mapping each state to a tuple (expr1, expr2) for gene1, gene2.
    gene1, gene2 : str
    vp : array-like
    out_dir : str
        Directory to save plots.
    min_cells : int
        Minimum number of cells required to proceed.
    scanning_window, resolution, angle_range : parameters for RSP computation.
    n_reps : int
        Number of permutations for null distribution.

    Returns
    -------
    stats : dict or None
        If enough cells: {
            'obs': {'CKD': float, 'AKI': float},
            'p': {'CKD': float, 'AKI': float},
            'null': np.ndarray of null RMSD values
        }
        Otherwise None.
    all_p : list[float]
        All p-values computed (two per disease state).
    """
    # ---- Prepare reference (Normal) ----
    expr1_norm, expr2_norm = expr["Normal"]

    # Determine expression thresholds for title annotations
    thr1_norm = np.quantile(expr1_norm, 1 - pct)
    thr2_norm = np.quantile(expr2_norm, 1 - pct)

    # Mask cells of this segment in Normal
    normal_obs = datasets["Normal"].obs["subclass.l2"].to_numpy() == seg
    bg_coords = polars["Normal"][normal_obs]
    bg_cart = coords_2d["Normal"][normal_obs]

    # Identify top-pct foreground masks
    fg1_mask = utils.select_top(expr1_norm, pct)[normal_obs]
    fg2_mask = utils.select_top(expr2_norm, pct)[normal_obs]

    # Extract angles and cartesian coords for foregrounds
    fg1_angle, fg2_angle = bg_coords[fg1_mask], bg_coords[fg2_mask]
    fg1_cart, fg2_cart = bg_cart[fg1_mask], bg_cart[fg2_mask]

    # Skip if too few cells
    if fg1_angle.size < min_cells or fg2_angle.size < min_cells:
        return None, []

    # Compute angular area curves for Normal
    a1_n, a2_n, bg_curve = core.compute_angular_area(
        theta_fg1=fg1_angle,
        theta_fg2=fg2_angle,
        theta_bg=bg_coords,
        scanning_window=scanning_window,
        resolution=resolution,
        scanning_range=angle_range,
        mode="absolute",
    )

    # Plot and save the Normal composite RSP
    core.plot_rsp_composite(
        bg_coords=bg_cart,
        fg_coords_dict={gene1: fg1_cart, gene2: fg2_cart},
        angle_range=angle_range,
        fg_rsp_dict={gene1: a1_n, gene2: a2_n},
        bg_curve=bg_curve,
        vantage_point=vp,
        vantage_point_color="black",
        vantage_point_label="Vantage Point",
        title=(
            f"Normal {seg} {gene1}/{gene2} @ top {int(pct*100)}% cells "
            f"({gene1}≥{thr1_norm:.3f}, {gene2}≥{thr2_norm:.3f})"
        ),
        save_path=os.path.join(
            out_dir, f"rsp_{seg}_{gene1}_{gene2}_Normal_top{int(pct*100)}pct.png"
        ),
    )

    # Reference concatenated areas for RMSD
    reference = np.concatenate([a1_n, a2_n])

    obs_vals, p_vals, all_p = {}, {}, []

    # ---- Loop over disease states ----
    for state in ("CKD", "AKI"):
        expr1_st, expr2_st = expr[state]
        thr1_st = np.quantile(expr1_st, 1 - pct)
        thr2_st = np.quantile(expr2_st, 1 - pct)

        state_obs = datasets[state].obs["subclass.l2"].to_numpy() == seg
        angles = polars[state][state_obs]
        coords = coords_2d[state][state_obs]

        mask1 = utils.select_top(expr1_st, pct)[state_obs]
        mask2 = utils.select_top(expr2_st, pct)[state_obs]
        fg1_s, fg2_s = angles[mask1], angles[mask2]
        cart1_s, cart2_s = coords[mask1], coords[mask2]

        # Compute RSP for this state
        a1_s, a2_s, bg_s = core.compute_angular_area(
            theta_fg1=fg1_s,
            theta_fg2=fg2_s,
            theta_bg=angles,
            scanning_window=scanning_window,
            resolution=resolution,
            scanning_range=angle_range,
            mode="absolute",
        )

        # Plot disease-state composite
        core.plot_rsp_composite(
            bg_coords=coords,
            fg_coords_dict={gene1: cart1_s, gene2: cart2_s},
            angle_range=angle_range,
            fg_rsp_dict={gene1: a1_s, gene2: a2_s},
            bg_curve=bg_s,
            vantage_point=vp,
            vantage_point_color="black",
            vantage_point_label="Vantage Point",
            title=(
                f"{state} {seg} {gene1}/{gene2} @ top {int(pct*100)}% cells "
                f"({gene1}≥{thr1_st:.3f}, {gene2}≥{thr2_st:.3f})"
            ),
            save_path=os.path.join(
                out_dir, f"rsp_{seg}_{gene1}_{gene2}_{state}_top{int(pct*100)}pct.png"
            ),
        )

        # Observed RMSD vs Normal
        obs = np.sqrt(np.mean((reference - np.concatenate([a1_s, a2_s])) ** 2))
        obs_vals[state] = obs

        # Build null distribution by permutation
        nulls = []
        idxs = np.arange(len(bg_coords))
        n1, n2 = fg1_angle.size, fg2_angle.size
        for _ in range(n_reps):
            perm = np.random.permutation(idxs)
            r1 = bg_coords[perm[:n1]]
            r2 = bg_coords[perm[n1 : n1 + n2]]
            a1_r, a2_r, _ = core.compute_angular_area(
                theta_fg1=r1,
                theta_fg2=r2,
                theta_bg=bg_coords,
                scanning_window=scanning_window,
                resolution=resolution,
                scanning_range=angle_range,
                mode="absolute",
            )
            nulls.append(
                np.sqrt(np.mean((reference - np.concatenate([a1_r, a2_r])) ** 2))
            )
        nulls = np.array(nulls)

        # Two-sided p-value
        lo = np.mean(nulls <= obs)
        hi = np.mean(nulls >= obs)
        p = min(2 * min(lo, hi), 1.0)
        p_vals[state] = p
        all_p.append(p)

    return {"obs": obs_vals, "p": p_vals, "null": nulls}, all_p

In [ ]:
gene_list = [
    ("STAT3", "TGFBR1"),
    ("ITGB6", "UMOD"),
    ("VCAM1", "STAT3"),
    ("VCAM1", "TGFBR1"),
    ("PROM1", "DCDC2"),
    ("KCTD1", "LCN2"),
    ("TARM1", "HAVCR1"),
    ("NRG1", "EGF"),
    ("NRG3", "EGF"),
]

# Percentile thresholds to sweep
percentiles = np.linspace(0.1, 0.9, 9)
n_reps = 1000  # Number of permutations
min_cells = 30  # Minimum cells to analyze a segment
scanning_window = np.pi
resolution = 100
angle_range = np.linspace(0, 2 * np.pi, resolution, endpoint=False)

# Map state names to AnnData objects
datasets = {
    "Normal": tal_normal_kpmp,
    "CKD": tal_ckd_kpmp,
    "AKI": tal_aki_kpmp,
}
subsegments = ["C-TAL", "M-TAL", "aTAL1", "aTAL2"]

for gene1, gene2 in gene_list:
    print(f"Processing genes: {gene1}, {gene2}")

    # Prepare output directory
    out_dir = f"figures/kpmp_joint_{gene1}_{gene2}"
    os.makedirs(out_dir, exist_ok=True)

    # Choose a shared vantage point from the combined data
    vp = np.asarray(
        utils.select_vantage_point(tal_kpmp, "X_umap", verbose=False)
    ).ravel()[:2]

    # Precompute coordinates and angles for each state
    coords_2d, polars = utils.load_coords_and_angles(datasets, vp)

    # Extract expression arrays for the two genes in each state
    expr = {
        state: (
            utils.extract_expr(ad, gene1),
            utils.extract_expr(ad, gene2),
        )
        for state, ad in datasets.items()
    }

    # Sweep over percentile thresholds
    for pct in percentiles:
        seg_stats = {}
        all_p = []
        # Compute stats for each subsegment
        for seg in subsegments:
            stats, pvals = compute_segment_stats(
                seg,
                pct,
                datasets,
                coords_2d,
                polars,
                expr,
                gene1,
                gene2,
                vp,
                out_dir,
                min_cells,
                scanning_window,
                resolution,
                angle_range,
                n_reps,
            )
            if stats is not None:
                seg_stats[seg] = stats
                all_p.extend(pvals)

        # Adjust all p-values across segments & states
        qvals = stats.bh_fdr(np.array(all_p))
        idx = 0
        # Assign q-values back into seg_stats
        for seg in subsegments:
            if seg in seg_stats:
                seg_stats[seg]["fdr_ckd"] = qvals[idx]
                seg_stats[seg]["fdr_aki"] = qvals[idx + 1]
                idx += 2
            else:
                # Fill with NaNs if skipped
                seg_stats[seg] = {
                    "fdr_ckd": np.nan,
                    "fdr_aki": np.nan,
                    "null": np.array([]),
                    "obs": {"CKD": np.nan, "AKI": np.nan},
                    "p": {"CKD": np.nan, "AKI": np.nan},
                }

        # Create 2×2 summary boxplot of null vs observed RMSD
        fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharey=True)
        for ax, seg in zip(axes.ravel(), subsegments):
            d = seg_stats[seg]
            if d["null"].size:
                # Plot null distribution
                ax.boxplot(
                    d["null"],
                    positions=[1],
                    widths=0.6,
                    patch_artist=True,
                    boxprops=dict(facecolor="burlywood", edgecolor="saddlebrown"),
                    medianprops=dict(color="black"),
                    whiskerprops=dict(color="saddlebrown"),
                    capprops=dict(color="saddlebrown"),
                    flierprops=dict(
                        marker="o", markerfacecolor="black", markersize=3, alpha=0.5
                    ),
                )
                # Overlay observed RMSD points
                ax.scatter(
                    1.1, d["obs"]["CKD"], color="red", s=30, label="N→CKD", zorder=3
                )
                ax.scatter(
                    1.2, d["obs"]["AKI"], color="blue", s=30, label="N→AKI", zorder=3
                )
                # Annotate p and q values with stars
                txt = (
                    f"p_CKD={d['p']['CKD']:.3f}, q_CKD={d['fdr_ckd']:.3f} {p_to_star(d['p']['CKD'])}\n"
                    f"p_AKI={d['p']['AKI']:.3f}, q_AKI={d['fdr_aki']:.3f} {p_to_star(d['p']['AKI'])}"
                )
                ax.text(
                    0.98,
                    0.98,
                    txt,
                    transform=ax.transAxes,
                    ha="right",
                    va="top",
                    fontsize=7,
                    bbox=dict(facecolor="white", edgecolor="gray", alpha=0.8),
                )
            else:
                # Indicate segment was skipped due to too few cells
                ax.text(
                    0.5,
                    0.5,
                    "skipped\n(n<%d)" % min_cells,
                    ha="center",
                    va="center",
                    fontsize=12,
                )
            ax.set_xticks([1])
            ax.set_xticklabels(["Null"])
            ax.set_title(seg)
            ax.legend(fontsize=8, loc="upper left")

        # Common labels and title
        for ax in axes.ravel():
            ax.set_ylim(0, 1)
            ax.set_ylabel("RMSD")
            ax.set_xlabel("Joint-gene foregrounds")
        plt.suptitle(
            f"Joint-gene RMSD @ top {int(pct*100)}% cells (equal N) — {gene1}/{gene2} — NULL={n_reps}",
            fontsize=14,
            y=0.95,
        )
        plt.tight_layout(rect=[0, 0, 1, 0.93])

        # Save the summary figure
        fn = os.path.join(out_dir, f"rmsd_eqN_{int(pct*100)}pct.png")
        plt.savefig(fn, dpi=300)
        plt.close(fig)

print("All done.")

This script plots UMAP embeddings of single-cell data across each condition, coloring each cell by its subtype. It arranges the three plots side by side, adds category-wise legends keyed to cell counts, and saves a high-resolution composite figure.

In [ ]:
datasets = [
    (normal_kpmp, f"Normal ({len(normal_kpmp)} cells)"),
    (ckd_kpmp, f"CKD ({len(ckd_kpmp)} cells)"),
    (aki_kpmp, f"AKI ({len(aki_kpmp)} cells)"),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("KPMP UMAP", fontsize=16)

# Adjust spacing to make room for the suptitle and between panels
plt.subplots_adjust(top=0.88, wspace=0.3)

# Iterate over each axis and its corresponding dataset
for ax, (adata, title) in zip(axes, datasets):
    tissues = adata.obs["subclass.l1"].astype("category")
    cats = tissues.cat.categories

    # Count cells per category, preserving category order
    counts = tissues.value_counts().reindex(cats)
    cmap = plt.get_cmap("tab20", len(cats))

    # Plot each category in descending order of abundance
    for i, cat in enumerate(counts.sort_values(ascending=False).index):
        mask = (tissues == cat).to_numpy()
        # UMAP coordinates for this subset
        x = adata.obsm["X_umap"][mask, 0]
        y = adata.obsm["X_umap"][mask, 1]

        ax.scatter(
            x,
            y,
            color=cmap(i),  # category-specific color
            s=1,  # very small points
            alpha=0.5,  # semi-transparent for overplotting
            label=cat,  # needed if one wanted auto-legend
        )

    # Build custom legend handles so we can list in category order
    handles = [
        Line2D([0], [0], marker="o", color=cmap(j), linestyle="", markersize=5)
        for j in range(len(cats))
    ]

    ax.legend(
        handles,
        counts.index,  # category labels in count order
        loc="upper right",  # legend location
        bbox_to_anchor=(1.15, 1.02),  # push it slightly outside plot
        fontsize=8,
        frameon=False,  # no box around legend
        title_fontsize=10,
        markerscale=2,  # make legend markers larger
        ncol=1,
        handletextpad=0.5,
        labelspacing=1,
        borderpad=0.5,
        handlelength=1.5,
        columnspacing=1.0,
    )

    ax.set_title(title)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/kpmp_umap.png", dpi=900)
plt.close(fig)

This script visualizes UMAP embeddings for the TAL subset of cells across each condition. It colors each cell by its finer-grained subclass, assembles the three plots side by side with consistent styling and legends, and saves a high-resolution figure.

In [ ]:
datasets = [
    (tal_normal_kpmp, f"Normal ({len(tal_normal_kpmp)} cells)"),
    (tal_ckd_kpmp, f"CKD ({len(tal_ckd_kpmp)} cells)"),
    (tal_aki_kpmp, f"AKI ({len(tal_aki_kpmp)} cells)"),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("KPMP UMAP (TAL)", fontsize=16)
plt.subplots_adjust(top=0.88, wspace=0.3)

for ax, (adata, title) in zip(axes, datasets):
    tissues = adata.obs["subclass.l2"].astype("category")
    cats = tissues.cat.categories

    counts = tissues.value_counts().reindex(cats)
    cmap = plt.get_cmap("tab20", len(cats))

    for i, cat in enumerate(counts.sort_values(ascending=False).index):
        mask = (tissues == cat).to_numpy()
        x = adata.obsm["X_umap"][mask, 0]
        y = adata.obsm["X_umap"][mask, 1]

        ax.scatter(
            x,
            y,
            color=cmap(i),
            s=1,
            alpha=0.5,
            label=cat,
        )

    handles = [
        Line2D([0], [0], marker="o", color=cmap(j), linestyle="", markersize=5)
        for j in range(len(cats))
    ]
    ax.legend(
        handles,
        counts.index,
        loc="upper right",
        bbox_to_anchor=(1.15, 1.02),
        fontsize=8,
        frameon=False,
        title_fontsize=10,
        markerscale=2,
        ncol=1,
        handletextpad=0.5,
        labelspacing=1,
        borderpad=0.5,
        handlelength=1.5,
        columnspacing=1.0,
    )

    ax.set_title(title)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/kpmp_umap_tal.png", dpi=900)
plt.close(fig)

This script generates and saves UMAP scatterplots for each TAL subclass across each condition. For each subclass, it creates a 1×3 panel showing only the cells of that subclass in each condition, annotates titles with cell counts, and writes out high-resolution figures.

In [ ]:
datasets = [
    (tal_normal_kpmp, "Normal"),
    (tal_ckd_kpmp, "CKD"),
    (tal_aki_kpmp, "AKI"),
]

out_dir = "figures/tal_umap"
os.makedirs(out_dir, exist_ok=True)

# Get the full set of TAL-level-2 subclass categories from the Normal data
all_l2 = tal_normal_kpmp.obs["subclass.l2"].astype("category").cat.categories

for l2_cat in all_l2:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle(f"KPMP UMAP ({l2_cat})", fontsize=16)
    plt.subplots_adjust(top=0.88, wspace=0.3)

    for ax, (adata, disease) in zip(axes, datasets):
        sel = adata.obs["subclass.l2"] == l2_cat
        sub = adata[sel].copy()

        ax.scatter(
            sub.obsm["X_umap"][:, 0],
            sub.obsm["X_umap"][:, 1],
            s=1,  # very small points
            alpha=0.5,
            color="C0",
        )

        # Annotate with the disease label and cell count
        ax.set_title(f"{disease} ({sub.n_obs} cells)")
        ax.set_xlabel("UMAP 1")
        ax.set_ylabel("UMAP 2")

    safe_cat = l2_cat.replace("/", "_")
    out_path = os.path.join(out_dir, f"tal_umap_{safe_cat}.png")

    plt.tight_layout()
    plt.savefig(out_path, dpi=900)
    plt.close(fig)

For a particular gene of interest, this script computes, for each TAL sub‐subclass, the percentage of cells exceeding a range of expression thresholds in each condition. It then plots coverage curves (percentage vs. threshold) and saves both the figures and underlying count tables.

In [ ]:
# Gene to threshold and the range of thresholds to sweep
gene_name = "UMOD"
thresholds = np.linspace(0, 3, num=100)

datasets = [
    (tal_normal_kpmp, "Normal"),
    (tal_ckd_kpmp, "CKD"),
    (tal_aki_kpmp, "AKI"),
]

desired_subcats = ["C-TAL", "M-TAL", "aTAL1", "aTAL2"]

out_dir = f"figures/tal_umap/{gene_name}"
os.makedirs(out_dir, exist_ok=True)

for adata, state in datasets:
    expr_matrix = adata[:, gene_name].X
    if isinstance(expr_matrix, (sparse.csr_matrix, sparse.csc_matrix)):
        expr = expr_matrix.toarray().ravel()
    else:
        expr = np.asarray(expr_matrix).ravel()

    full_obs = adata.obs["subclass.l2"]

    # Total cell count per desired subclass (zero if absent)
    totals = full_obs.value_counts().reindex(desired_subcats, fill_value=0)

    # Prepare array: rows=subclasses, cols=thresholds
    counts = np.zeros((len(desired_subcats), len(thresholds)), dtype=float)

    for j, thr in enumerate(thresholds):
        mask = expr > thr
        high_obs = full_obs[mask]
        vc = high_obs.value_counts()
        for i, cat in enumerate(desired_subcats):
            # Avoid division by zero by using 1 when a subclass has no cells
            denom = totals[cat] if totals[cat] else 1
            counts[i, j] = vc.get(cat, 0) * 100.0 / denom

    # --- Plot coverage curves ---
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, cat in enumerate(desired_subcats):
        ax.plot(thresholds, counts[i], linewidth=1.5, label=cat)

    ax.set_title(f"{state} TAL ({gene_name}) Thresholding", fontsize=16)
    ax.set_xlabel("Expression Threshold")
    ax.set_ylabel("Coverage (%)")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(
        fontsize=8,
        title="subclass.l2",
        frameon=False,
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
    )

    plt.tight_layout()
    out_path = os.path.join(
        out_dir, f"threshold_counts_{state.lower()}_{gene_name}.png"
    )
    plt.savefig(out_path, dpi=900)
    plt.close(fig)

    counts_df = pd.DataFrame(
        counts, index=desired_subcats, columns=[f"{thr:.2f}" for thr in thresholds]
    )
    counts_df.index.name = "Subclass"
    csv_path = os.path.join(
        out_dir, f"threshold_counts_{state.lower()}_{gene_name}.csv"
    )
    counts_df.to_csv(csv_path)

    print(f"Saved counts for {state} TAL with {gene_name} thresholds to {out_path}")

This code defines a helper to compute, for each TAL subclass, the percentage coverage above various expression thresholds, then calculates and plots fold‐changes in coverage between each condition for the gene of interest. It saves both the fold‐change curves and the corresponding tables.

In [ ]:
# Tiny constant to avoid division by zero in fold‐change
eps = 1e-6


def get_coverage_matrix(adata):
    """
    Compute coverage (%) of cells above each threshold, for each TAL subclass.

    Parameters
    ----------
    adata : AnnData
        Single-cell dataset containing expression in .X and 'subclass.l2' in .obs.

    Returns
    -------
    np.ndarray, shape (n_subclasses, n_thresholds)
        coverage percentages for desired_subcats across thresholds.
    """
    # Extract raw expression vector for the target gene
    mat = adata[:, gene_name].X
    expr = mat.toarray().ravel() if sparse.issparse(mat) else np.asarray(mat).ravel()

    obs = adata.obs["subclass.l2"]
    mask_total = obs.isin(desired_subcats)
    filtered_expr = expr[mask_total]
    filtered_obs = obs[mask_total]

    cov = np.zeros((len(desired_subcats), len(thresholds)), dtype=float)
    totals = filtered_obs.value_counts().reindex(desired_subcats, fill_value=0)

    # For each threshold, compute % of cells above threshold per subclass
    for j, thr in enumerate(thresholds):
        sel = filtered_expr > thr
        vc = filtered_obs[sel].value_counts()
        for i, cat in enumerate(desired_subcats):
            denom = totals[cat] if totals[cat] else 1
            cov[i, j] = vc.get(cat, 0) * 100.0 / denom

    return cov


normal_cov = get_coverage_matrix(tal_normal_kpmp)

for adata, state in datasets[1:]:
    # Compute coverage for this condition
    disease_cov = get_coverage_matrix(adata)

    # Fold‐change relative to Normal (with eps to stabilize)
    fc = (disease_cov + eps) / (normal_cov + eps)

    fig, ax = plt.subplots(figsize=(10, 6))
    for i, cat in enumerate(desired_subcats):
        ax.plot(thresholds, fc[i], linewidth=1.5, label=cat)

    ax.set_title(f"{state} vs Normal fold-change ({gene_name})", fontsize=16)
    ax.set_xlabel("Expression Threshold")
    ax.set_ylabel("Fold-change")
    ax.axhline(1, color="gray", linestyle="--", linewidth=1)  # unity reference
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(
        fontsize=8,
        frameon=False,
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
    )

    plt.tight_layout()
    png_path = os.path.join(out_dir, f"fc_{state.lower()}_{gene_name}.png")
    plt.savefig(png_path, dpi=900)
    plt.close(fig)

    fc_df = pd.DataFrame(
        fc, index=desired_subcats, columns=[f"{thr:.2f}" for thr in thresholds]
    )
    fc_df.index.name = "Subclass"
    csv_path = os.path.join(out_dir, f"fc_{state.lower()}_{gene_name}.csv")
    fc_df.to_csv(csv_path)

    print(f"Saved fold-change for {state} to {png_path} and {csv_path}")